# getitrack: Multi-Object Tracking Demo

`getitrack` is a lightweight multi-object tracking toolkit. This notebook runs the
**ByteTrack** algorithm on a synthetic scene and shows the tracked output inline.

The scene feeds the tracker realistic, imperfect detections (position noise, jittered
confidence scores, and random dropouts), so the demo exercises the real algorithm:

- **Stable identities** — each object keeps one id across the whole clip.
- **Two-stage association** — low-confidence detections still update tracks.
- **Coasting and recovery** — a dropped detection does not lose the id.

In [ ]:
from pathlib import Path

import cv2
import numpy as np

import getitrack
from getitrack import BaseTracker, ByteTrackConfig, TrackAnnotator, to_h264
from getitrack.core.detection import Detections
from getitrack.io import VideoWriter

getitrack.__version__

## 1. Configure the tracker

`ByteTrackConfig` holds the ByteTrack parameters; `BaseTracker.from_config` resolves
the `algorithm` name to the matching tracker.

In [ ]:
config = ByteTrackConfig(verbose=False)
tracker = BaseTracker.from_config(config)
type(tracker).__name__, config.high_score_threshold, config.match_threshold

## 2. Synthetic scene

Boxes move at constant velocity and bounce off the edges. `detections()` returns
detector-like output: the true boxes with corner noise, jittered scores, and a 5%
chance of dropout per object.

In [ ]:
W, H, N, FRAMES, SEED = 960, 540, 6, 200, 0


def reset() -> None:
    global rng, sizes, tl, vel
    rng = np.random.default_rng(SEED)
    sizes = rng.uniform(0.08, 0.14, (N, 2)) * min(W, H)
    tl = rng.uniform(0, 1, (N, 2)) * (np.array([W, H]) - sizes)
    vel = rng.uniform(-5, 5, (N, 2))


def step() -> None:
    global tl, vel
    tl = tl + vel
    hi = np.array([W, H]) - sizes
    for a in range(2):
        out = (tl[:, a] < 0) | (tl[:, a] > hi[:, a])
        vel[out, a] *= -1
        tl[:, a] = np.clip(tl[:, a], 0, hi[:, a])


def detections(frame_id):
    gt = np.concatenate([tl, tl + sizes], axis=1)
    keep = rng.random(N) >= 0.05  # 5% dropout
    n = int(keep.sum())
    boxes = gt[keep] + rng.normal(0, 1.5, (n, 4))  # corner noise
    scores = np.clip(rng.normal(0.85, 0.12, n), 0.05, 0.99)  # score jitter
    dets = Detections(boxes.astype(np.float32), scores.astype(np.float32), np.zeros(n, np.int64), frame_id)
    return dets, boxes

## 3. Run tracking and write an annotated video

Each frame: advance the scene, build detections, call `tracker.update`, draw the raw
detections in gray and the tracked boxes (colored by id) with `TrackAnnotator`.

In [ ]:
reset()
tracker = BaseTracker.from_config(config)
annotator = TrackAnnotator(show_score=True)
output = Path("getitrack_demo.mp4")
active_by_frame = []

with VideoWriter(output, fps=30.0, frame_size=(W, H)) as writer:
    for frame_id in range(FRAMES):
        step()
        dets, raw = detections(frame_id)
        tracked = tracker.update(dets)
        frame = np.full((H, W, 3), 28, np.uint8)
        for x1, y1, x2, y2 in raw.astype(int):
            cv2.rectangle(frame, (x1, y1), (x2, y2), (90, 90, 90), 1)  # raw detections
        writer.write(annotator.annotate(frame, tracked))
        active_by_frame.append({int(t) for t in tracked.track_ids})

video_path = to_h264(output, output.with_name("getitrack_demo_h264.mp4"))
print(f"{writer.frames_written} frames, {len(set().union(*active_by_frame))} ids -> {video_path}")

## 4. Watch the result

Gray = raw detections fed in. Colored boxes with ids = tracker output.

In [ ]:
from IPython.display import Video

Video(str(video_path), embed=True, width=760)

## 5. Identity timeline

Each row is one track id; a filled cell means the id was active on that frame. Short
gaps are the tracker coasting through a dropped detection and recovering the same id.

In [ ]:
import matplotlib.pyplot as plt

ids = sorted(set().union(*active_by_frame))
grid = np.array([[i in frame for frame in active_by_frame] for i in ids], dtype=int)

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.imshow(grid, aspect="auto", cmap="Greens", interpolation="nearest")
ax.set_yticks(range(len(ids)))
ax.set_yticklabels([f"id {i}" for i in ids])
ax.set_xlabel("frame")
ax.set_title("Active track per frame (gaps = coasting through a dropped detection)")
plt.tight_layout()
plt.show()